In [8]:
import sys
import os
# Add parent directory to sys.path
parent_dir = os.path.abspath("..")
sys.path.append(parent_dir)
from pipeline import (
    process_vocab_word
)

import json
import requests
from constants import (
    ANKI_CONNECT_URL    
)

In [9]:
# Function to send the card to Anki
def add_card_to_anki(deck_name, vocab_data):
    """
    Sends the formatted Anki card to Anki using AnkiConnect API.
    """
    note = {
        "deckName": deck_name,
        "modelName": "Basic", 
        "fields": {
            "Front": f"{vocab_data['vocab_word']}\n\n{vocab_data['example_sentence']}",
            "Back": f"{vocab_data['vocab_translation']}\n\n{vocab_data['example_sentence_translation']}",
        },
        "audio": [
            {"url": vocab_data["vocab_audio"], "filename": "vocab_word.mp3", "fields": ["Front"]},
            {"url": vocab_data["example_sentence_translation_audio"], "filename": "example_sentence.mp3", "fields": ["Back"]}
        ]
    }
    
    payload = {"action": "addNote", "version": 6, "params": {"note": note}}
    
    response = requests.post(ANKI_CONNECT_URL, json=payload).json()
    return response

In [10]:
def request(action, **params):
    return {'action': action, 'params': params, 'version': 6}

def invoke(action, **params):
    payload = request(action, **params)
    response = requests.post(ANKI_CONNECT_URL, json=payload)
    
    if not response.ok:
        raise Exception(f"Request failed with status code {response.status_code}: {response.text}")
    
    response_json = response.json()
    
    if 'error' not in response_json or 'result' not in response_json:
        raise Exception('Invalid response structure')
    if response_json['error'] is not None:
        raise Exception(response_json['error'])
    
    return response_json['result']

In [11]:
import base64

def store_audio_file(filename):
    abs_path = os.path.abspath(filename)  # Get absolute path of the file
    with open(abs_path, "rb") as f:
        audio_data = base64.b64encode(f.read()).decode("utf-8")  # Encode to Base64 and decode to a string
    
    print(abs_path)
    
    response = invoke("storeMediaFile", filename=filename, path=abs_path)
    print(f"Stored {filename}: {response}")  # Print Anki Connect response

In [12]:
# invoke('deckNames')

In [13]:
invoke('createDeck', deck='test1')

1741653962153

In [14]:
# invoke('deleteDecks', decks="test2", cardsToo=True)

In [15]:
# invoke('modelNames')

In [16]:
# print my vocab mining card type
invoke("modelTemplates", modelName="Vocab Mining")

{'Card 1': {'Front': '<span style="font-size: 50px;">{{Target Word}}</span><br>\n{{Word Audio}}',
  'Back': '{{FrontSide}}\n<hr id=answer>\n<span style="font-size: 35px;">{{Source Translation}}</span><br>\n{{Word Audio}}{{Sentence Audio}}<br>\n<span style="font-size: 25px;">{{Target Sentence}}<br></span>\n{{Source Sentence Translation}}<br>\n'},
 'Card 2': {'Front': '<span style="font-size: 50px;">{{Source Translation}}</span>',
  'Back': '{{FrontSide}}\n\n<hr id=answer>\n<span style="font-size: 35px;">{{Target Word}}</span><br>\n{{Word Audio}}{{Sentence Audio}}<br>\n<span style="font-size: 25px;">{{Target Sentence}}<br></span>\n{{Source Sentence Translation}}<br>\n'}}

In [17]:
invoke("modelFieldNames", modelName="Vocab Mining")

['Target Word',
 'Source Translation',
 'Target Sentence',
 'Source Sentence Translation',
 'Word Audio',
 'Sentence Audio']

In [18]:
# generate test fields
"""
{
    "vocab_word": vocab_word,
    "vocab_translation": vocab_translation,
    "example_sentence": example_sentence,
    "example_sentence_translation": example_sentence_translation,
    "vocab_audio": vocab_audio,
    "example_sentence_translation_audio": example_sentence_audio
}
"""
# todo: infer language based on input?
result = process_vocab_word("猫", target_language="Japanese")


Processing vocabulary word: 猫

Checking if audio files exist...
猫_word.mp3: True
猫_sentence.mp3: True


In [19]:
result

{'vocab_word': '猫',
 'vocab_translation': 'Cat',
 'example_sentence': '私の猫は黒いです。',
 'example_sentence_translation': 'My cat is black.',
 'vocab_audio_filename': '猫_word.mp3',
 'example_sentence_translation_audio_filename': '猫_sentence.mp3'}

In [20]:
vocab_word = result["vocab_word"]
vocab_translation = result["vocab_translation"]
example_sentence = result["example_sentence"]
example_sentence_translation = result["example_sentence_translation"]
vocab_audio_filename = result["vocab_audio_filename"]
example_sentence_translation_audio_filename = result["example_sentence_translation_audio_filename"]

In [21]:
store_audio_file(vocab_audio_filename)
store_audio_file(example_sentence_translation_audio_filename)

/Users/sethdonaldson/sourcecode/anki-vocab-generator/notebook/猫_word.mp3
Stored 猫_word.mp3: 猫_word.mp3
/Users/sethdonaldson/sourcecode/anki-vocab-generator/notebook/猫_sentence.mp3
Stored 猫_sentence.mp3: 猫_sentence.mp3


In [22]:
vocab_audio_filename

'猫_word.mp3'

In [23]:
import os

print("Checking if audio files exist...")
print(f"{vocab_audio_filename}: {os.path.exists(vocab_audio_filename)}")
print(f"{example_sentence_translation_audio_filename}: {os.path.exists(example_sentence_translation_audio_filename)}")

Checking if audio files exist...
猫_word.mp3: True
猫_sentence.mp3: True


In [24]:

deck_name = "test1"
model_name = "Vocab Mining"
fields = {
    'Target Word': vocab_word,
    'Source Translation': vocab_translation,
    'Target Sentence': example_sentence,
    'Source Sentence Translation':example_sentence_translation,
}
audio = [
    {
        "filename": vocab_audio_filename,
        "path": os.path.abspath(vocab_audio_filename),
        "fields": [
            "Word Audio"
        ]
    },
    {
        "filename": example_sentence_translation_audio_filename,
        "path": os.path.abspath(example_sentence_translation_audio_filename),
        "fields": [
            "Sentence Audio"
        ]
    }
]
note = {
    "deckName": deck_name,
    "modelName": model_name,
    "fields": fields,
    "audio": audio,
    "tags": ["stenchtoast"],
    "options": {
            "allowDuplicate": False,
            "duplicateScope": "deck",
            "duplicateScopeOptions": {
                "deckName": deck_name,
                "checkChildren": False,
                "checkAllModels": False
            }
    }
}

In [25]:
note

{'deckName': 'test1',
 'modelName': 'Vocab Mining',
 'fields': {'Target Word': '猫',
  'Source Translation': 'Cat',
  'Target Sentence': '私の猫は黒いです。',
  'Source Sentence Translation': 'My cat is black.'},
 'audio': [{'filename': '猫_word.mp3',
   'path': '/Users/sethdonaldson/sourcecode/anki-vocab-generator/notebook/猫_word.mp3',
   'fields': ['Word Audio']},
  {'filename': '猫_sentence.mp3',
   'path': '/Users/sethdonaldson/sourcecode/anki-vocab-generator/notebook/猫_sentence.mp3',
   'fields': ['Sentence Audio']}],
 'tags': ['stenchtoast'],
 'options': {'allowDuplicate': False,
  'duplicateScope': 'deck',
  'duplicateScopeOptions': {'deckName': 'test1',
   'checkChildren': False,
   'checkAllModels': False}}}

In [26]:
# add card to deck
invoke("addNote", note=note)

1741653992126